# Analyze Extracted Skills (Post-Processing)

**This notebook analyzes already-extracted skill data from parquet files.**

✅ **Benefits:**
- No need to re-run extraction (saves hours)
- Can analyze results even if extraction was interrupted
- Quick iteration on different analyses
- Works with partial results

**Use case:** You've already run `extract_skills_from_jd.ipynb` and have results in a parquet file.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import json
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
sns.set_palette('husl')

print("✓ Imports successful")

## Configuration

**Edit these paths to match your data:**

In [ ]:
# =====================================================================
# EDIT THESE PATHS
# =====================================================================

# Input: Your extracted skills parquet file
INPUT_PATH = '../data/jd_extracted_skills.parquet'

# Column names (adjust if your parquet has different column names)
SKILLS_COLUMN = 'skills'              # List of extracted skills
NUM_SKILLS_COLUMN = 'num_skills'      # Number of skills
BY_SECTION_COLUMN = 'by_section'      # Skills grouped by section (dict)
ONET_CODE_COLUMN = 'onet_code'        # ONET occupation code
DATE_COLUMN = 'post_date'             # Job posting date
JD_TEXT_COLUMN = 'job_description'    # Original JD text (if available)

# Analysis options
TOP_N_SKILLS = 20                     # Show top N most common skills
TOP_N_ONET = 10                       # Show top N ONET codes

# =====================================================================

print(f"Configuration:")
print(f"  Input: {INPUT_PATH}")
print(f"  Top skills to show: {TOP_N_SKILLS}")
print(f"  Top ONET codes to show: {TOP_N_ONET}")

## Step 1: Load Extracted Skills Data

Load the parquet file that contains already-extracted skills.

In [ ]:
# Check if file exists
if not Path(INPUT_PATH).exists():
    print(f"❌ File not found: {INPUT_PATH}")
    print(f"\nMake sure you've run the extraction notebook first!")
    print(f"Expected file location: {Path(INPUT_PATH).absolute()}")
else:
    print(f"Loading extracted skills from {INPUT_PATH}...")
    df = pd.read_parquet(INPUT_PATH)
    
    print(f"\n✓ Loaded {len(df):,} job descriptions with extracted skills")
    print(f"\nFile size: {Path(INPUT_PATH).stat().st_size / 1024**2:.1f} MB")
    print(f"\nColumns: {list(df.columns)}")
    
    # Show sample
    print(f"\nSample data:")
    display(df.head(3))

## Step 2: Data Quality Check

In [ ]:
print("="*70)
print("DATA QUALITY CHECK")
print("="*70)

# Basic info
print(f"\nTotal records: {len(df):,}")
print(f"Date range: {df[DATE_COLUMN].min()} to {df[DATE_COLUMN].max()}" if DATE_COLUMN in df.columns else "No date column")

# Check for missing values
print(f"\nMissing values:")
for col in [SKILLS_COLUMN, NUM_SKILLS_COLUMN, ONET_CODE_COLUMN]:
    if col in df.columns:
        null_count = df[col].isna().sum()
        null_pct = (null_count / len(df)) * 100
        print(f"  {col}: {null_count:,} ({null_pct:.1f}%)")

# Skills extraction success rate
if NUM_SKILLS_COLUMN in df.columns:
    jds_with_skills = (df[NUM_SKILLS_COLUMN] > 0).sum()
    jds_without_skills = (df[NUM_SKILLS_COLUMN] == 0).sum()
    
    print(f"\nSkills extraction:")
    print(f"  JDs with skills: {jds_with_skills:,} ({jds_with_skills/len(df)*100:.1f}%)")
    print(f"  JDs without skills: {jds_without_skills:,} ({jds_without_skills/len(df)*100:.1f}%)")

## Step 3: Overall Statistics

In [ ]:
print("="*70)
print("OVERALL STATISTICS")
print("="*70)

# Skills per JD statistics
if NUM_SKILLS_COLUMN in df.columns:
    skills_stats = df[NUM_SKILLS_COLUMN].describe()
    
    print(f"\nSkills per Job Description:")
    print(f"  Mean:   {skills_stats['mean']:.1f}")
    print(f"  Median: {skills_stats['50%']:.1f}")
    print(f"  Std:    {skills_stats['std']:.1f}")
    print(f"  Min:    {skills_stats['min']:.0f}")
    print(f"  Max:    {skills_stats['max']:.0f}")
    print(f"  25%:    {skills_stats['25%']:.1f}")
    print(f"  75%:    {skills_stats['75%']:.1f}")

# Total unique skills
if SKILLS_COLUMN in df.columns:
    all_skills = []
    for skills_list in df[SKILLS_COLUMN].dropna():
        if isinstance(skills_list, list):
            all_skills.extend(skills_list)
    
    unique_skills = len(set(all_skills))
    total_skill_mentions = len(all_skills)
    
    print(f"\nSkill diversity:")
    print(f"  Total unique skills: {unique_skills:,}")
    print(f"  Total skill mentions: {total_skill_mentions:,}")
    print(f"  Avg mentions per skill: {total_skill_mentions/unique_skills:.1f}")

## Step 4: Most Common Skills

Analyze the most frequently mentioned skills across all job descriptions.

In [ ]:
# Count skill frequencies
if SKILLS_COLUMN in df.columns:
    all_skills = []
    for skills_list in df[SKILLS_COLUMN].dropna():
        if isinstance(skills_list, list):
            all_skills.extend(skills_list)
    
    skill_counter = Counter(all_skills)
    top_skills = skill_counter.most_common(TOP_N_SKILLS)
    
    print("="*70)
    print(f"TOP {TOP_N_SKILLS} MOST COMMON SKILLS")
    print("="*70)
    
    for rank, (skill, count) in enumerate(top_skills, 1):
        pct = (count / len(df)) * 100
        print(f"{rank:2d}. {skill:50s} {count:6,} JDs ({pct:5.1f}%)")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 8))
    skills_names = [s[0] for s in top_skills]
    skills_counts = [s[1] for s in top_skills]
    
    ax.barh(range(len(skills_names)), skills_counts, color='steelblue')
    ax.set_yticks(range(len(skills_names)))
    ax.set_yticklabels(skills_names)
    ax.set_xlabel('Number of Job Descriptions')
    ax.set_title(f'Top {TOP_N_SKILLS} Most Common Skills')
    ax.invert_yaxis()
    
    # Add count labels
    for i, count in enumerate(skills_counts):
        ax.text(count, i, f' {count:,}', va='center')
    
    plt.tight_layout()
    plt.show()

## Step 5: Skills Distribution

Visualize the distribution of number of skills per job description.

In [ ]:
if NUM_SKILLS_COLUMN in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(df[NUM_SKILLS_COLUMN], bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(df[NUM_SKILLS_COLUMN].mean(), color='red', linestyle='--', 
                    label=f'Mean: {df[NUM_SKILLS_COLUMN].mean():.1f}')
    axes[0].axvline(df[NUM_SKILLS_COLUMN].median(), color='green', linestyle='--', 
                    label=f'Median: {df[NUM_SKILLS_COLUMN].median():.1f}')
    axes[0].set_xlabel('Number of Skills')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Skills per Job Description')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    axes[1].boxplot(df[NUM_SKILLS_COLUMN].dropna(), vert=True)
    axes[1].set_ylabel('Number of Skills')
    axes[1].set_title('Skills per JD - Box Plot')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Percentiles
    print("\nPercentile breakdown:")
    for p in [10, 25, 50, 75, 90, 95, 99]:
        value = np.percentile(df[NUM_SKILLS_COLUMN].dropna(), p)
        print(f"  {p}th percentile: {value:.1f} skills")

## Step 6: Skills by KSAO Section

Analyze skill distribution across KSAO categories (Knowledge, Skills, Abilities, Other).

In [ ]:
if BY_SECTION_COLUMN in df.columns:
    # Count skills by section
    section_counts = {}
    section_jd_counts = {}  # How many JDs have skills in each section
    
    for by_section in df[BY_SECTION_COLUMN].dropna():
        if isinstance(by_section, dict):
            for section, skills in by_section.items():
                if section not in section_counts:
                    section_counts[section] = 0
                    section_jd_counts[section] = 0
                section_counts[section] += len(skills)
                section_jd_counts[section] += 1
    
    print("="*70)
    print("SKILLS BY KSAO SECTION")
    print("="*70)
    
    for section in sorted(section_counts.keys(), key=lambda x: section_counts[x], reverse=True):
        total = section_counts[section]
        jd_count = section_jd_counts[section]
        avg_per_jd = total / jd_count if jd_count > 0 else 0
        print(f"  {section:25s}: {total:7,} total, {jd_count:6,} JDs, {avg_per_jd:.1f} avg/JD")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Total skills by section
    sections = list(section_counts.keys())
    counts = [section_counts[s] for s in sections]
    
    axes[0].bar(sections, counts, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('KSAO Section')
    axes[0].set_ylabel('Total Skills Extracted')
    axes[0].set_title('Total Skills by KSAO Category')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Average skills per JD by section
    avgs = [section_counts[s] / section_jd_counts[s] for s in sections]
    
    axes[1].bar(sections, avgs, color='coral', edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('KSAO Section')
    axes[1].set_ylabel('Average Skills per JD')
    axes[1].set_title('Average Skills per JD by KSAO Category')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## Step 7: Analysis by ONET Code

Compare skill requirements across different occupations.

In [ ]:
if ONET_CODE_COLUMN in df.columns and NUM_SKILLS_COLUMN in df.columns:
    # Top ONET codes by frequency
    top_onet_codes = df[ONET_CODE_COLUMN].value_counts().head(TOP_N_ONET)
    
    print("="*70)
    print(f"TOP {TOP_N_ONET} ONET CODES BY FREQUENCY")
    print("="*70)
    
    for onet_code, count in top_onet_codes.items():
        pct = (count / len(df)) * 100
        avg_skills = df[df[ONET_CODE_COLUMN] == onet_code][NUM_SKILLS_COLUMN].mean()
        print(f"  {onet_code}: {count:6,} JDs ({pct:5.1f}%), avg {avg_skills:.1f} skills")
    
    # Skills per JD by ONET code
    df_top_onet = df[df[ONET_CODE_COLUMN].isin(top_onet_codes.index)]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    df_top_onet.boxplot(column=NUM_SKILLS_COLUMN, by=ONET_CODE_COLUMN, ax=ax)
    ax.set_xlabel('ONET Code')
    ax.set_ylabel('Number of Skills')
    ax.set_title(f'Skills per JD by ONET Code (Top {TOP_N_ONET})')
    plt.suptitle('')  # Remove default title
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Step 8: Time Series Analysis (if date available)

Analyze how skill requirements change over time.

In [ ]:
if DATE_COLUMN in df.columns and NUM_SKILLS_COLUMN in df.columns:
    # Convert to datetime and create quarter column
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors='coerce')
    df['quarter'] = df[DATE_COLUMN].dt.to_period('Q')
    
    # Filter out rows with invalid dates
    df_with_dates = df[df['quarter'].notna()]
    
    if len(df_with_dates) > 0:
        quarterly_stats = df_with_dates.groupby('quarter')[NUM_SKILLS_COLUMN].agg([
            'count', 'mean', 'median', 'std'
        ])
        
        print("="*70)
        print("TIME SERIES ANALYSIS")
        print("="*70)
        print(f"\nQuarterly statistics:")
        display(quarterly_stats)
        
        # Visualize
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        
        # Average skills over time
        x_labels = quarterly_stats.index.astype(str)
        
        axes[0].plot(x_labels, quarterly_stats['mean'], marker='o', linewidth=2, label='Mean')
        axes[0].plot(x_labels, quarterly_stats['median'], marker='s', linewidth=2, label='Median')
        axes[0].fill_between(range(len(x_labels)), 
                             quarterly_stats['mean'] - quarterly_stats['std'],
                             quarterly_stats['mean'] + quarterly_stats['std'],
                             alpha=0.2, label='±1 Std Dev')
        axes[0].set_xlabel('Quarter')
        axes[0].set_ylabel('Skills per JD')
        axes[0].set_title('Average Skills per Job Description Over Time')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')
        
        # Sample size over time
        axes[1].bar(x_labels, quarterly_stats['count'], color='steelblue', edgecolor='black', alpha=0.7)
        axes[1].set_xlabel('Quarter')
        axes[1].set_ylabel('Number of Job Descriptions')
        axes[1].set_title('Sample Size per Quarter')
        axes[1].grid(True, alpha=0.3)
        plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ No valid dates found in the data")
else:
    print("⚠️ Date column not available for time series analysis")

## Step 9: Skill Co-occurrence Analysis

Find which skills frequently appear together in job descriptions.

In [ ]:
if SKILLS_COLUMN in df.columns:
    # Analyze co-occurrence for top skills
    top_5_skills = [s[0] for s in skill_counter.most_common(5)]
    
    print("="*70)
    print("SKILL CO-OCCURRENCE ANALYSIS")
    print("="*70)
    print(f"\nAnalyzing co-occurrence for top 5 skills...\n")
    
    for target_skill in top_5_skills:
        # Find JDs that have this skill
        jds_with_skill = df[df[SKILLS_COLUMN].apply(
            lambda x: target_skill in x if isinstance(x, list) else False
        )]
        
        # Count other skills that appear with this skill
        co_occurring_skills = []
        for skills_list in jds_with_skill[SKILLS_COLUMN]:
            if isinstance(skills_list, list):
                co_occurring_skills.extend([s for s in skills_list if s != target_skill])
        
        co_occur_counter = Counter(co_occurring_skills)
        top_co_occur = co_occur_counter.most_common(5)
        
        print(f"Skills that commonly appear with '{target_skill}':")
        for skill, count in top_co_occur:
            pct = (count / len(jds_with_skill)) * 100
            print(f"  - {skill:40s} {count:5,} ({pct:5.1f}%)")
        print()

## Step 10: Export Summary Report

Save a summary of the analysis to a JSON file.

In [ ]:
# Create summary report
summary = {
    'generated_at': datetime.now().isoformat(),
    'input_file': INPUT_PATH,
    'total_jds': len(df),
    'date_range': {
        'start': str(df[DATE_COLUMN].min()) if DATE_COLUMN in df.columns else None,
        'end': str(df[DATE_COLUMN].max()) if DATE_COLUMN in df.columns else None
    },
    'skills_per_jd': {
        'mean': float(df[NUM_SKILLS_COLUMN].mean()) if NUM_SKILLS_COLUMN in df.columns else None,
        'median': float(df[NUM_SKILLS_COLUMN].median()) if NUM_SKILLS_COLUMN in df.columns else None,
        'std': float(df[NUM_SKILLS_COLUMN].std()) if NUM_SKILLS_COLUMN in df.columns else None,
        'min': float(df[NUM_SKILLS_COLUMN].min()) if NUM_SKILLS_COLUMN in df.columns else None,
        'max': float(df[NUM_SKILLS_COLUMN].max()) if NUM_SKILLS_COLUMN in df.columns else None
    },
    'total_unique_skills': unique_skills if SKILLS_COLUMN in df.columns else None,
    'top_skills': [(skill, int(count)) for skill, count in top_skills[:10]] if SKILLS_COLUMN in df.columns else [],
    'skills_by_section': {section: int(count) for section, count in section_counts.items()} if BY_SECTION_COLUMN in df.columns else {},
    'top_onet_codes': [(str(code), int(count)) for code, count in top_onet_codes.items()] if ONET_CODE_COLUMN in df.columns else []
}

# Save to JSON
output_json = INPUT_PATH.replace('.parquet', '_analysis_summary.json')
with open(output_json, 'w') as f:
    json.dump(summary, f, indent=2)

print("="*70)
print("SUMMARY REPORT SAVED")
print("="*70)
print(f"\n✓ Saved analysis summary to: {output_json}")
print(f"\nYou can reload this summary anytime without re-running the analysis.")

## Custom Analysis Section

Use this section for your own custom analyses.

In [ ]:
# Your custom analysis code here
# Example: Filter by specific ONET code
# df_software_engineers = df[df[ONET_CODE_COLUMN] == '15-1252.00']

# Example: Find emerging skills (increasing over time)
# ...

# Example: Compare skills across industries
# ...